# Lahore PM2.5 AQI (OpenAQ v3)
## Export: point GeoJSON at monitor resolution


### 0. Initialize Imports and API Access


In [9]:
import calendar
import math
import os
import time
from pathlib import Path

import geopandas as gpd
import pandas as pd
import requests
from shapely.geometry import Point


def load_env_file(path):
    if not path.exists():
        return
    for raw_line in path.read_text().splitlines():
        line = raw_line.strip()
        if not line or line.startswith("#") or "=" not in line:
            continue
        key, value = line.split("=", 1)
        os.environ.setdefault(key.strip(), value.strip().strip('\"').strip("'"))


for env_path in [Path(".env"), Path("notebooks/AQI/.env")]:
    load_env_file(env_path)

OPENAQ_BASE = "https://api.openaq.org/v3"
PARAMETER_ID_PM25 = 2
API_KEY = os.getenv("OPENAQ_API_KEY") or os.getenv("API_KEY")

if not API_KEY:
    raise RuntimeError("Missing OPENAQ_API_KEY or API_KEY in the environment or notebooks/AQI/.env.")


### 1. Parameters


In [10]:
UC_SHP = "../../data/Lahore UCs/Lahore UC.shp"
YEAR = 2026
MONTH = 2
REQUEST_SLEEP_S = 1.0
MAX_RETRIES = 6
BACKOFF_BASE_S = 2.0

last_day = calendar.monthrange(YEAR, MONTH)[1]
START = f"{YEAR:04d}-{MONTH:02d}-01"
END = f"{YEAR:04d}-{MONTH:02d}-{last_day:02d}"
OUT_PREFIX = f"AQI_Lahore_{calendar.month_abbr[MONTH]}{YEAR}"

print(f"Using date range: {START} to {END}")
print(f"Output prefix: {OUT_PREFIX}")


Using date range: 2026-02-01 to 2026-02-28
Output prefix: AQI_Lahore_Feb2026


### 2. Load Lahore Boundary and Fetch Monthly PM2.5 Sensor Means


In [11]:
gdf = gpd.read_file(UC_SHP)
if gdf.crs is None:
    gdf = gdf.set_crs(4326)
if gdf.crs.to_epsg() != 4326:
    gdf = gdf.to_crs(4326)
gdf["geometry"] = gdf["geometry"].buffer(0)

minx, miny, maxx, maxy = gdf.total_bounds
bbox = f"{minx:.4f},{miny:.4f},{maxx:.4f},{maxy:.4f}"


session = requests.Session()
request_cache = {}


def openaq_get(path, params=None, max_retries=MAX_RETRIES):
    headers = {"X-API-Key": API_KEY}
    params = params or {}
    cache_key = (path, tuple(sorted(params.items())))
    if cache_key in request_cache:
        return request_cache[cache_key]

    response = None
    for attempt in range(max_retries):
        response = session.get(
            f"{OPENAQ_BASE}{path}",
            params=params,
            headers=headers,
            timeout=60,
        )

        if response.status_code == 429:
            retry_after = response.headers.get("Retry-After")
            try:
                wait_s = max(float(retry_after), REQUEST_SLEEP_S) if retry_after else 0.0
            except (TypeError, ValueError):
                wait_s = 0.0
            if wait_s <= 0:
                wait_s = max(REQUEST_SLEEP_S, BACKOFF_BASE_S * (2 ** attempt))
            print(f"Rate limited on {path}; sleeping {wait_s:.1f}s before retry {attempt + 1}/{max_retries}.")
            time.sleep(wait_s)
            continue

        response.raise_for_status()
        payload = response.json()
        request_cache[cache_key] = payload
        return payload

    response.raise_for_status()
    return response.json()


def fetch_pm25_locations(bbox_string, sleep_s=0.15):
    page = 1
    rows = []

    while True:
        payload = openaq_get(
            "/locations",
            params={
                "bbox": bbox_string,
                "parameters_id": PARAMETER_ID_PM25,
                "limit": 100,
                "page": page,
            },
        )
        results = payload.get("results", [])
        if not results:
            break

        for loc in results:
            rows.append({
                "location_id": loc.get("id"),
                "location": loc.get("name"),
            })

        found = int((payload.get("meta", {}) or {}).get("found") or 0)
        if page * 100 >= found:
            break
        page += 1
        time.sleep(sleep_s)

    return pd.DataFrame(rows).drop_duplicates(subset=["location_id"])


def fetch_sensor_monthly_mean(sensor_id, start_date, end_date, sleep_s=0.15):
    payload = openaq_get(
        f"/sensors/{sensor_id}/days/monthly",
        params={
            "date_from": start_date,
            "date_to": end_date,
            "limit": 100,
            "page": 1,
        },
    )
    values = [rec.get("value") for rec in payload.get("results", []) if rec.get("value") is not None]
    time.sleep(sleep_s)
    if not values:
        return (math.nan, 0)
    series = pd.Series(values, dtype="float64")
    return (float(series.mean()), int(series.notna().sum()))


def fetch_pm25_sensor_points(locations_df, start_date, end_date, sleep_s=0.15):
    rows = []

    for _, loc in locations_df.iterrows():
        loc_id = int(loc["location_id"])
        sensors_payload = openaq_get(f"/locations/{loc_id}/sensors")

        for sensor in sensors_payload.get("results", []):
            parameter = sensor.get("parameter") or {}
            if parameter.get("id") != PARAMETER_ID_PM25:
                continue

            sensor_id = sensor.get("id")
            latest = sensor.get("latest") or {}
            coords = latest.get("coordinates") or {}
            lat = coords.get("latitude")
            lon = coords.get("longitude")

            if sensor_id is None or lat is None or lon is None:
                continue

            monthly_mean, months_returned = fetch_sensor_monthly_mean(
                sensor_id,
                start_date,
                end_date,
                sleep_s=sleep_s,
            )
            if pd.isna(monthly_mean):
                continue

            rows.append({
                "location": loc["location"],
                "location_id": loc_id,
                "sensor_id": sensor_id,
                "latitude": lat,
                "longitude": lon,
                "pm25_monthly_avg": monthly_mean,
                "pm25_unit": parameter.get("units") or "µg/m³",
                "months_returned": months_returned,
            })

        time.sleep(sleep_s)

    if not rows:
        return gpd.GeoDataFrame(
            pd.DataFrame(
                columns=[
                    "location",
                    "location_id",
                    "sensor_id",
                    "latitude",
                    "longitude",
                    "pm25_monthly_avg",
                    "pm25_unit",
                    "months_returned",
                ]
            ),
            geometry=[],
            crs="EPSG:4326",
        )

    df = pd.DataFrame(rows).drop_duplicates(subset=["sensor_id"])
    return gpd.GeoDataFrame(
        df,
        geometry=[Point(xy) for xy in zip(df["longitude"], df["latitude"])],
        crs="EPSG:4326",
    )


PM25_BREAKS = [
    (0.0, 12.0, 0, 50, "Good"),
    (12.1, 35.4, 51, 100, "Moderate"),
    (35.5, 55.4, 101, 150, "Unhealthy for Sensitive Groups"),
    (55.5, 150.4, 151, 200, "Unhealthy"),
    (150.5, 250.4, 201, 300, "Very Unhealthy"),
    (250.5, 350.4, 301, 400, "Hazardous"),
    (350.5, 500.4, 401, 500, "Hazardous"),
]


def pm25_to_us_epa_aqi(concentration):
    if pd.isna(concentration):
        return (math.nan, None)
    for c_low, c_high, i_low, i_high, category in PM25_BREAKS:
        if c_low <= concentration <= c_high:
            aqi = ((i_high - i_low) / (c_high - c_low)) * (concentration - c_low) + i_low
            return (round(aqi), category)
    return (500, "Hazardous")


### 3. Export Point GeoJSON


In [12]:
locations = fetch_pm25_locations(bbox, sleep_s=REQUEST_SLEEP_S)
points = fetch_pm25_sensor_points(locations, START, END, sleep_s=REQUEST_SLEEP_S)

if points.empty:
    raise RuntimeError("No monthly PM2.5 sensor data returned from OpenAQ for the selected month and Lahore bbox.")

aqi_pairs = points["pm25_monthly_avg"].apply(pm25_to_us_epa_aqi)
points["pm25_aqi"] = [pair[0] for pair in aqi_pairs]
points["pm25_aqi_category"] = [pair[1] for pair in aqi_pairs]
points["month"] = f"{YEAR:04d}-{MONTH:02d}"

geojson_path = f"{OUT_PREFIX}_points.geojson"
points.to_file(geojson_path, driver="GeoJSON")

print(f"Locations fetched: {len(locations)}")
print(f"Sensors exported: {len(points)}")
print(f"Saved: {geojson_path}")


Locations fetched: 76
Sensors exported: 64
Saved: AQI_Lahore_Feb2026_points.geojson
